In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import os
from google.colab import drive #type:ignore
import torch

drive.mount(r'/content/drive/')
df = pd.read_csv(r"/content/drive/MyDrive/DL_CSV/fmnist_small.csv")
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

# Checking for the availability of GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

x = x / 255.0

# Creating custom class dataset
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28) #(batch size,channels,row,column)
        self.labels = torch.tensor(labels, dtype=torch.long) 

    def __len__(self):
        return len(self.features)     

    def __getitem__(self, index):
        return self.features[index], self.labels[index]
    
# Performing DataSet Split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, shuffle=True)

# Creating CustomDataset Object for train and test
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# Creating DataLoader object of the class
# Note: num_workers=2 works fine, but if Colab ever throws a broken pipe error, change it to 0.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,num_workers=0)

# Creating our CNN and ANN
class ArtificialNN(nn.Module):
    def __init__(self, input_channels):
        super().__init__()

        #For CNN
        self.features=nn.Sequential(
            #For first pair of conv. layer
            nn.Conv2d(input_channels,out_channels=32,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            #For 2nd Pair
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )

        #Creating the ANN
        self.classifier=nn.Sequential(
            #Flattening the tensor
            nn.Flatten(),

            #Making the first hidden layer 
            nn.Linear(64*7*7,out_features=128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            #Making hte second hidden layer
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64,10)
        )

    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)

        return x


# Defining Epochs and Learning Rate
epochs = 100
learning_rate = 0.1

model = ArtificialNN(input_channels=1)
model = model.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# Performing Training
for i in range(epochs):
    total_loss = 0

    for features, label in train_loader:
        features, label = features.to(device), label.to(device)
        y_pred = model(features)

        # Making gradient zero
        optimizer.zero_grad()

        # Calculating Loss
        loss = loss_function(y_pred, label)

        # Backward
        loss.backward()

        # Updating Parameters
        optimizer.step()

        total_loss = total_loss + loss.item()
    
    print(f"For epoch :{i+1}, Loss={total_loss/len(train_loader)*100:.2f}%")

# Evaluating the model
model.eval() 
total = 0
correct = 0

with torch.no_grad():
    for features, labels in test_loader:
        # FIX: Changed labels(device) to labels.to(device)
        features, labels = features.to(device), labels.to(device)
        y_pred = model(features)

        _, predicted = torch.max(y_pred, 1)
        total = total + labels.shape[0]

        correct = correct + (predicted == labels).sum().item()

print(f"Accuracy of Model = {(correct/total)*100:.2f}%")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Using device: cuda
For epoch :1, Loss=175.29%
For epoch :2, Loss=107.74%
For epoch :3, Loss=87.61%
For epoch :4, Loss=75.38%
For epoch :5, Loss=66.83%
For epoch :6, Loss=62.63%
For epoch :7, Loss=57.54%
For epoch :8, Loss=55.40%
For epoch :9, Loss=51.98%
For epoch :10, Loss=50.63%
For epoch :11, Loss=47.83%
For epoch :12, Loss=44.45%
For epoch :13, Loss=42.80%
For epoch :14, Loss=40.79%
For epoch :15, Loss=39.19%
For epoch :16, Loss=38.48%
For epoch :17, Loss=35.01%
For epoch :18, Loss=34.25%
For epoch :19, Loss=32.21%
For epoch :20, Loss=30.92%
For epoch :21, Loss=30.88%
For epoch :22, Loss=29.27%
For epoch :23, Loss=27.08%
For epoch :24, Loss=26.67%
For epoch :25, Loss=24.97%
For epoch :26, Loss=25.49%
For epoch :27, Loss=23.73%
For epoch :28, Loss=22.77%
For epoch :29, Loss=22.32%
For epoch :30, Loss=20.84%
For epoch :31, Loss=21.15%
For epoch :32, Loss=